# One-dimensional axial-radius comparison

Translate the MicroBooNE dipole and $z$-expansion extractions, the deuterium dipole and $z$-expansion results, and the MINERvA and LQCD $z$-expansion results into the common observable

$$r_A^2=-\frac{6}{g_A}\left.\frac{dF_A}{dQ^2}\right|_{Q^2=0}.$$

The transformation is applied sample by sample, retaining coefficient correlations and non-Gaussian posterior shapes. Values are reported in $\mathrm{fm}^2$. The intervals below are central equal-tailed 68% (approximately one-standard-deviation) credible/confidence intervals.

In [ ]:
from pathlib import Path
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Locate this repository whether the notebook is launched here or from axial_mass.
start = Path.cwd().resolve()
helper_dir = next(
    (parent / 'ma_zexp' / 'python' / 'scripts' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'python' / 'scripts' / 'postfit_physical_parameters.py').is_file()),
    None,
)
if helper_dir is None:
    raise FileNotFoundError('Could not locate ma_zexp/python/scripts/postfit_physical_parameters.py')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from postfit_physical_parameters import (
    FA_SOURCE_COLORS, PUBLICATION_RC, SPECS, _reference_prior_samples, load_fit,
)

HBARC_GEV_FM = 0.1973269804
SUITES = (
    ('nuwro_fit_results', 'NuWro'),
    ('asimov_fit_results', 'Asimov'),
    ('opendata_fit_results', 'Open data'),
)
MICROBOONE_MA_FIT = 'ma_uniform'
MICROBOONE_ZEXP_FIT = 'minerva_k6_uniform'
REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2 = False  # diagnostic truncation only
N_REFERENCE_SAMPLES = 200_000
RANDOM_SEED = 2026
CONFIDENCE_LEVELS = (0.68,)


mpl.rcParams.update(PUBLICATION_RC)


## Transformation

For a dipole, $F_A(Q^2)=g_A(1+Q^2/M_A^2)^{-2}$, so $r_A^2=12/M_A^2$. For $F_A=\sum_k a_k z^k$, the derivative is evaluated analytically using the basis metadata belonging to each result. The factor $(\hbar c)^2$ converts $\mathrm{GeV}^{-2}$ to $\mathrm{fm}^2$.

In [ ]:
def radius_squared_from_ma(ma_gev):
    ma = np.asarray(ma_gev, dtype=float)
    if np.any(~np.isfinite(ma)) or np.any(ma <= 0):
        raise ValueError('Every M_A sample must be finite and positive')
    return 12.0 / ma**2 * HBARC_GEV_FM**2


def equivalent_ma_from_radius_squared(radius_squared_fm2):
    """Dipole M_A with the same Q2=0 slope; defined only for r_A^2 > 0."""
    radius_squared = np.asarray(radius_squared_fm2, dtype=float)
    if np.any(~np.isfinite(radius_squared)) or np.any(radius_squared <= 0):
        raise ValueError('Every r_A^2 sample must be finite and positive')
    return np.sqrt(12.0 * HBARC_GEV_FM**2 / radius_squared)


def dz_dq2_at_zero(t0_gev2, t_cut_gev2):
    # z=(sqrt(tcut+Q2)-sqrt(tcut-t0))/(sqrt(tcut+Q2)+sqrt(tcut-t0))
    a = np.sqrt(t_cut_gev2)
    b = np.sqrt(t_cut_gev2 - t0_gev2)
    return b / (a * (a + b)**2)


def radius_squared_from_zexp(coefficients, t0_gev2, t_cut_gev2, g_a):
    coefficients = np.atleast_2d(np.asarray(coefficients, dtype=float))
    z0 = ((np.sqrt(t_cut_gev2) - np.sqrt(t_cut_gev2 - t0_gev2)) /
          (np.sqrt(t_cut_gev2) + np.sqrt(t_cut_gev2 - t0_gev2)))
    k = np.arange(1, coefficients.shape[1])
    dfa_dq2 = (coefficients[:, 1:] @ (k * z0**(k - 1))) * dz_dq2_at_zero(
        t0_gev2, t_cut_gev2
    )
    return -6.0 / g_a * dfa_dq2 * HBARC_GEV_FM**2


def transform_a1_to_t0_zero(coefficients, t0_gev2, t_cut_gev2):
    """Return the exact coefficient of z_new about Q2=0 (t0_new=0)."""
    coefficients = np.atleast_2d(np.asarray(coefficients, dtype=float))
    # z_old = (z_new + c)/(1 + c*z_new), with c=z_old(Q2=0).
    c = ((np.sqrt(t_cut_gev2) - np.sqrt(t_cut_gev2 - t0_gev2)) /
         (np.sqrt(t_cut_gev2) + np.sqrt(t_cut_gev2 - t0_gev2)))
    k = np.arange(1, coefficients.shape[1])
    return (1.0 - c**2) * (coefficients[:, 1:] @ (k * c**(k - 1)))


def radius_squared_from_t0_zero_a1(coefficients, t0_gev2, t_cut_gev2, g_a):
    a1_t0_zero = transform_a1_to_t0_zero(coefficients, t0_gev2, t_cut_gev2)
    # For t0=0, z(0)=0 and dz/dQ2|0 = 1/(4*t_cut).
    return -3.0 * a1_t0_zero / (2.0 * g_a * t_cut_gev2) * HBARC_GEV_FM**2


def central_interval(samples, probability):
    tail = (1.0 - probability) / 2.0
    return np.quantile(samples, [tail, 0.5, 1.0 - tail])


# Numerical cross-checks of both analytic transformations.
assert np.isclose(radius_squared_from_ma([1.0])[0] / HBARC_GEV_FM**2, 12.0)
test_coefficients, test_t0, test_tcut = np.array([[1.0, -2.0, 0.5]]), -0.5, 0.2
eps = 1e-7
def test_fa(q2):
    z = ((np.sqrt(test_tcut + q2) - np.sqrt(test_tcut - test_t0)) /
         (np.sqrt(test_tcut + q2) + np.sqrt(test_tcut - test_t0)))
    return np.sum(test_coefficients[0] * z**np.arange(test_coefficients.shape[1]))
numeric_slope = (test_fa(eps) - test_fa(0.0)) / eps
analytic_r2 = radius_squared_from_zexp(test_coefficients, test_t0, test_tcut, -1.27)[0]
basis_r2 = radius_squared_from_t0_zero_a1(
    test_coefficients, test_t0, test_tcut, -1.27
)[0]
assert np.isclose(analytic_r2, -6 / -1.27 * numeric_slope * HBARC_GEV_FM**2, rtol=2e-6)
assert np.isclose(analytic_r2, basis_r2, rtol=2e-14, atol=2e-14)

## Build the six one-dimensional distributions

`ma_uniform` and `minerva_k6_uniform` are used for the MicroBooNE points because their axial parameters have no Gaussian pull penalty. Change the two keys above to compare a different fit variant. Setting `REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2=True` discards MicroBooNE $z$-expansion MCMC samples with $r_A^2<0$. This is a diagnostic conditional distribution, not a refit or a physical prior, and the retained fraction is reported explicitly. The deuterium dipole input is $M_A=1.014\pm0.014$ GeV. Every $z$-expansion result shown uses the common $k_{\max}=6$, $t_0=-0.50\ \mathrm{GeV}^2$, $t_{\mathrm{cut}}=9(0.134\ \mathrm{GeV})^2$, and $F_A(0)=-1.2754$ convention. The deuterium input is the published result translated to this common basis; MINERvA and LQCD use their correlated $k_{\max}=6$ coefficient covariance matrices.

In [ ]:
def build_suite_results(suite):
    spec_by_key = {spec.key: spec for spec in SPECS}
    ma_result = load_fit(spec_by_key[MICROBOONE_MA_FIT], suite)
    zexp_result = load_fit(spec_by_key[MICROBOONE_ZEXP_FIT], suite)
    if ma_result is None or zexp_result is None:
        raise FileNotFoundError(f'Missing a requested fit output in {suite}')

    rng = np.random.default_rng(RANDOM_SEED)
    deuterium_ma = rng.normal(1.014, 0.014, N_REFERENCE_SAMPLES)
    deuterium_coeff, deuterium_t0, deuterium_tcut = _reference_prior_samples(
        'deuterium_k6', N_REFERENCE_SAMPLES, RANDOM_SEED + 1
    )
    minerva_coeff, minerva_t0, minerva_tcut = _reference_prior_samples(
        'minerva_k6', N_REFERENCE_SAMPLES, RANDOM_SEED + 2
    )
    lqcd_coeff, lqcd_t0, lqcd_tcut = _reference_prior_samples(
        'lqcd_k6', N_REFERENCE_SAMPLES, RANDOM_SEED + 3
    )
    zprior = zexp_result['spec'].prior

    microboone_zexp_coeff = zexp_result['samples']
    microboone_zexp_r2_raw = radius_squared_from_zexp(
        microboone_zexp_coeff, zprior.t0_gev2, zprior.t_cut_gev2, zprior.fa_q2_zero
    )
    positive_r2_mask = microboone_zexp_r2_raw >= 0.0
    positive_r2_fraction = positive_r2_mask.mean()
    if REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2:
        if not positive_r2_mask.any():
            raise ValueError('No MicroBooNE z-expansion samples have non-negative r_A^2')
        microboone_zexp_coeff = microboone_zexp_coeff[positive_r2_mask]
        microboone_zexp_r2 = microboone_zexp_r2_raw[positive_r2_mask]
        microboone_zexp_label = (
            r'MicroBooNE $z$ expansion ($k_{\max}=6$, $r_A^2\geq0$ diagnostic)'
        )
    else:
        microboone_zexp_r2 = microboone_zexp_r2_raw
        microboone_zexp_label = r'MicroBooNE $z$ expansion ($k_{\max}=6$)'

    diagnostic_status = pd.DataFrame({
        'positive-r_A^2 requirement enabled': [REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2],
        'total MicroBooNE z-expansion samples': [len(microboone_zexp_r2_raw)],
        'retained samples': [len(microboone_zexp_r2)],
        'positive-r_A^2 fraction': [positive_r2_fraction],
    })
    display(diagnostic_status.style.format({'positive-r_A^2 fraction': '{:.3%}'}))

    distributions = {
        r'MicroBooNE dipole $M_A$': radius_squared_from_ma(ma_result['samples'][:, 0]),
        microboone_zexp_label: microboone_zexp_r2,
        r'Deuterium dipole $M_A$': radius_squared_from_ma(deuterium_ma),
        r'Deuterium $z$ expansion ($k_{\max}=6$)': radius_squared_from_zexp(
            deuterium_coeff, deuterium_t0, deuterium_tcut, -1.2754
        ),
        r'MINERvA $z$ expansion ($k_{\max}=6$)': radius_squared_from_zexp(
            minerva_coeff, minerva_t0, minerva_tcut, -1.2754
        ),
        r'LQCD $z$ expansion ($k_{\max}=6$)': radius_squared_from_zexp(
            lqcd_coeff, lqcd_t0, lqcd_tcut, -1.2754
        ),
    }

    # Independent double-check: transform to the t0=0 basis, where the radius
    # depends on a1 alone. Agreement is tested for every individual sample.
    zexp_inputs = {
        microboone_zexp_label: (
            microboone_zexp_coeff, zprior.t0_gev2, zprior.t_cut_gev2, zprior.fa_q2_zero
        ),
        r'Deuterium $z$ expansion ($k_{\max}=6$)': (
            deuterium_coeff, deuterium_t0, deuterium_tcut, -1.2754
        ),
        r'MINERvA $z$ expansion ($k_{\max}=6$)': (
            minerva_coeff, minerva_t0, minerva_tcut, -1.2754
        ),
        r'LQCD $z$ expansion ($k_{\max}=6$)': (
            lqcd_coeff, lqcd_t0, lqcd_tcut, -1.2754
        ),
    }
    basis_check_records = []
    for label, (coefficients, t0, tcut, g_a) in zexp_inputs.items():
        direct = distributions[label]
        transformed = radius_squared_from_t0_zero_a1(coefficients, t0, tcut, g_a)
        difference = transformed - direct
        basis_check_records.append({
            'result': label,
            'median transformed a1 (t0=0)': np.median(
                transform_a1_to_t0_zero(coefficients, t0, tcut)
            ),
            'median r_A^2 direct': np.median(direct),
            'median r_A^2 via t0=0': np.median(transformed),
            'max abs sample difference': np.max(np.abs(difference)),
        })
        assert np.allclose(transformed, direct, rtol=2e-13, atol=2e-13)
    basis_check = pd.DataFrame(basis_check_records).set_index('result')
    display(basis_check.style.format('{:.12g}'))

    records = []
    for label, samples in distributions.items():
        record = {'result': label, 'mean': np.mean(samples), 'std': np.std(samples, ddof=1)}
        for probability in CONFIDENCE_LEVELS:
            low, median, high = central_interval(samples, probability)
            record.update({
                'median': median, f'low_{probability:.2f}': low, f'high_{probability:.2f}': high
            })
        records.append(record)
    summary = pd.DataFrame(records).set_index('result')
    display(summary.style.format('{:.4f}'))

    equivalent_ma_distributions = {}
    equivalent_ma_records = []
    for label, radius_samples in distributions.items():
        positive = radius_samples > 0.0
        if not positive.any():
            raise ValueError(f'{label} has no positive r_A^2 samples to map to M_A')
        ma_samples = equivalent_ma_from_radius_squared(radius_samples[positive])
        equivalent_ma_distributions[label] = ma_samples
        low, median, high = central_interval(ma_samples, 0.68)
        equivalent_ma_records.append({
            'result': label,
            'positive-r_A^2 fraction': positive.mean(),
            'retained samples': positive.sum(),
            'total samples': len(radius_samples),
            'mean M_A [GeV]': np.mean(ma_samples),
            'std M_A [GeV]': np.std(ma_samples, ddof=1),
            'median': median, 'low_0.68': low, 'high_0.68': high,
        })
    equivalent_ma_summary = pd.DataFrame(equivalent_ma_records).set_index('result')
    display(equivalent_ma_summary.style.format({
        'positive-r_A^2 fraction': '{:.3%}',
        'mean M_A [GeV]': '{:.4f}', 'std M_A [GeV]': '{:.4f}',
        'median': '{:.4f}', 'low_0.68': '{:.4f}', 'high_0.68': '{:.4f}',
    }))
    return {
        'distributions': distributions,
        'summary': summary,
        'diagnostic_status': diagnostic_status,
        'basis_check': basis_check,
        'equivalent_ma_distributions': equivalent_ma_distributions,
        'equivalent_ma_summary': equivalent_ma_summary,
    }


## Common comparison

The first forest and density plots compare $r_A^2$. The second pair applies the inverse dipole relation $M_A^{\mathrm{equiv}}=\sqrt{12(\hbar c)^2/r_A^2}$, giving the dipole mass with the same slope at $Q^2=0$. This inverse is real only for $r_A^2>0$, so non-positive samples are excluded and the retained fraction is reported explicitly. All forest plots show central 68% intervals; density panels are normalized to unit area.

In [ ]:
def plot_suite_results(suite, distributions, summary):
    colors = ['#0072B2', '#1607E4', '#4D4D4D', FA_SOURCE_COLORS['deuterium'],
              FA_SOURCE_COLORS['minerva_k6'], FA_SOURCE_COLORS['lqcd_k6']]
    labels = list(distributions)
    y = np.arange(len(labels))[::-1]

    interval_fig, ax_interval = plt.subplots(
        figsize=(7.2, 4.8), constrained_layout=True
    )
    for yi, (label, color) in enumerate(zip(labels, colors)):
        row = summary.loc[label]
        plot_y = y[yi]
        low, high = row['low_0.68'], row['high_0.68']
        ax_interval.errorbar(
            row['median'], plot_y,
            xerr=[[row['median'] - low], [high - row['median']]],
            fmt='o', color=color, markersize=6.5, elinewidth=2.2,
            capsize=4, capthick=1.5, zorder=3,
        )

    ax_interval.set_yticks(y, labels)
    ax_interval.set_xlabel(r'$r_A^2\ [\mathrm{fm}^2]$')
    ax_interval.grid(axis='x', color='#9AA4B2', alpha=0.18, linewidth=0.7)
    ax_interval.tick_params(axis='y', left=False, right=False)
    ax_interval.minorticks_on()
    ax_interval.text(
        0.02, 0.03, 'Central 68% intervals', transform=ax_interval.transAxes,
        color='#596273', fontsize=11, ha='left', va='bottom',
    )

    all_low = min(np.quantile(values, 0.001) for values in distributions.values())
    all_high = max(np.quantile(values, 0.999) for values in distributions.values())
    density_fig, ax_density = plt.subplots(
        figsize=(7.2, 5.2), constrained_layout=True
    )
    for (label, samples), color in zip(distributions.items(), colors):
        counts, edges = np.histogram(samples, bins=180, range=(all_low, all_high), density=True)
        centers = (edges[1:] + edges[:-1]) / 2
        kernel_x = np.arange(-8, 9)
        kernel = np.exp(-0.5 * (kernel_x / 2.0)**2); kernel /= kernel.sum()
        density = np.convolve(counts, kernel, mode='same')
        ax_density.plot(centers, density, color=color, linewidth=2, label=label)

    ax_density.set_xlabel(r'$r_A^2\ [\mathrm{fm}^2]$')
    ax_density.set_ylabel('Probability density')
    ax_density.grid(color='#9AA4B2', alpha=0.18, linewidth=0.7)
    ax_density.minorticks_on()
    ax_density.legend(
        fontsize=10, handlelength=2.3, labelspacing=0.45, frameon=False
    )

    suffix = '_positive_r2_diagnostic' if REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2 else ''
    figure_dir = helper_dir.parents[1] / 'figs' / suite
    figure_dir.mkdir(parents=True, exist_ok=True)
    for figure, stem in (
        (interval_fig, figure_dir / f'axial_radius_intervals{suffix}'),
        (density_fig, figure_dir / f'axial_radius_densities{suffix}'),
    ):
        figure.savefig(stem.with_suffix('.pdf'), dpi=600, bbox_inches='tight',
                       pad_inches=0.03, facecolor='white')
        figure.savefig(stem.with_suffix('.png'), dpi=600, bbox_inches='tight',
                       pad_inches=0.03, facecolor='white')
    csv_name = f'axial_radius_comparison{suffix}.csv'
    summary.to_csv(figure_dir / csv_name)
    display(interval_fig)
    display(density_fig)
    plt.close(interval_fig)
    plt.close(density_fig)
    return interval_fig, density_fig


def plot_equivalent_ma_results(suite, distributions, summary):
    colors = ['#0072B2', '#1607E4', '#4D4D4D', FA_SOURCE_COLORS['deuterium'],
              FA_SOURCE_COLORS['minerva_k6'], FA_SOURCE_COLORS['lqcd_k6']]
    labels = list(distributions)
    y = np.arange(len(labels))[::-1]

    interval_fig, ax_interval = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
    for yi, (label, color) in enumerate(zip(labels, colors)):
        row = summary.loc[label]
        ax_interval.errorbar(
            row['median'], y[yi],
            xerr=[[row['median'] - row['low_0.68']],
                  [row['high_0.68'] - row['median']]],
            fmt='o', color=color, markersize=6.5, elinewidth=2.2,
            capsize=4, capthick=1.5, zorder=3,
        )
    ax_interval.set_yticks(y, labels)
    ax_interval.set_xlabel(r'Equivalent dipole $M_A$ [GeV]')
    ax_interval.set_xlim(0.0, 2.0)
    ax_interval.grid(axis='x', color='#9AA4B2', alpha=0.18, linewidth=0.7)
    ax_interval.tick_params(axis='y', left=False, right=False)
    ax_interval.minorticks_on()
    ma_plot_range = (0.0, 2.0)
    linear_edges = np.linspace(*ma_plot_range, 181)
    density_fig, ax_density = plt.subplots(figsize=(7.2, 5.2), constrained_layout=True)
    for (label, samples), color in zip(distributions.items(), colors):
        counts, edges = np.histogram(samples, bins=linear_edges)
        centers = (edges[1:] + edges[:-1]) / 2
        kernel_x = np.arange(-8, 9)
        kernel = np.exp(-0.5 * (kernel_x / 2.0)**2)
        kernel /= kernel.sum()
        density = counts / (len(samples) * np.diff(edges))
        density = np.convolve(density, kernel, mode='same')
        ax_density.plot(centers, density, color=color, linewidth=2, label=label)
    ax_density.set_xlabel(r'Equivalent dipole $M_A$ [GeV]')
    ax_density.set_ylabel('Probability density')
    ax_density.set_xlim(ma_plot_range)
    ax_density.grid(color='#9AA4B2', alpha=0.18, linewidth=0.7)
    ax_density.minorticks_on()
    ax_density.legend(
        fontsize=10, handlelength=2.3, labelspacing=0.45, frameon=False
    )

    suffix = '_positive_r2_diagnostic' if REQUIRE_POSITIVE_MICROBOONE_ZEXP_R2 else ''
    figure_dir = helper_dir.parents[1] / 'figs' / suite
    for figure, stem in (
        (interval_fig, figure_dir / f'equivalent_ma_intervals{suffix}'),
        (density_fig, figure_dir / f'equivalent_ma_densities{suffix}'),
    ):
        figure.savefig(stem.with_suffix('.pdf'), dpi=600, bbox_inches='tight',
                       pad_inches=0.03, facecolor='white')
        figure.savefig(stem.with_suffix('.png'), dpi=600, bbox_inches='tight',
                       pad_inches=0.03, facecolor='white')
    summary.to_csv(figure_dir / f'equivalent_ma_comparison{suffix}.csv')
    display(interval_fig)
    display(density_fig)
    plt.close(interval_fig)
    plt.close(density_fig)
    return interval_fig, density_fig


In [ ]:
suite_outputs = {}
for suite, suite_label in SUITES:
    print(f'\n=== {suite_label}: {suite} ===')
    suite_result = build_suite_results(suite)
    suite_result['figures'] = plot_suite_results(
        suite, suite_result['distributions'], suite_result['summary']
    )
    suite_result['equivalent_ma_figures'] = plot_equivalent_ma_results(
        suite, suite_result['equivalent_ma_distributions'],
        suite_result['equivalent_ma_summary'],
    )
    suite_outputs[suite] = suite_result
